<a href="https://colab.research.google.com/github/mhtrannn/Dynamic-Customer-Segmentation-/blob/main/Online_Retail_II_DataPrep_CusID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

file_path = "/content/online_retail_II.xlsx"

excel_file = pd.ExcelFile(file_path)

print(excel_file.sheet_names)

['Year 2009-2010', 'Year 2010-2011']


In [ ]:
df_2009_2010 = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010"
)

df_2010_2011 = pd.read_excel(
    file_path,
    sheet_name="Year 2010-2011"
)

print("2009-2010:", df_2009_2010.shape)
print("2010-2011:", df_2010_2011.shape)

2009-2010: (525461, 8)
2010-2011: (541910, 8)


In [ ]:
df = pd.concat(
    [df_2009_2010, df_2010_2011],
    ignore_index=True
)

print("Kích thước dataset:", df.shape)

Kích thước dataset: (1067371, 8)


In [ ]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [ ]:
df.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680.0,France


In [ ]:
df.columns.tolist()

['Invoice',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'Price',
 'Customer ID',
 'Country']

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Start date:", df["InvoiceDate"].min())
print("End date:", df["InvoiceDate"].max())

Start date: 2009-12-01 07:45:00
End date: 2011-12-09 12:50:00


In [ ]:
df["Customer ID"].isna().sum()

np.int64(243007)

In [ ]:
missing_customer = df["Customer ID"].isna().sum()

missing_percent = (
    missing_customer / len(df) * 100
)

print("Số dòng thiếu CustomerID:", missing_customer)
print("Tỷ lệ:", round(missing_percent, 2), "%")

Số dòng thiếu CustomerID: 243007
Tỷ lệ: 22.77 %


In [ ]:
df = df.dropna(subset=["Customer ID"])
print("Dataset size after removing missing Customer ID:", df.shape)

Dataset size after removing missing Customer ID: (824364, 8)


In [ ]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 26479


In [ ]:
df = df.drop_duplicates()
print("Dataset size after removing duplicates:", df.shape)

Dataset size after removing duplicates: (797885, 8)


kiem tra cancellation

In [ ]:
df["Invoice"] = df["Invoice"].astype(str)

cancellation = df["Invoice"].str.upper().str.startswith("C")

print("Cancellation transactions:", cancellation.sum())

Cancellation transactions: 18390


In [ ]:
negative_quantity = (df["Quantity"] < 0).sum()

print("Transactions with negative Quantity:", negative_quantity)

Transactions with negative Quantity: 18390


In [ ]:
negative_non_cancellation = (
    (df["Quantity"] < 0) &
    (~cancellation)
).sum()

print("Negative Quantity without cancellation:", negative_non_cancellation)

Negative Quantity without cancellation: 0


In [ ]:
df = df[~cancellation]

print("Dataset size after removing cancellations:", df.shape)

Dataset size after removing cancellations: (779495, 8)


In [ ]:
print("Quantity <= 0:", (df["Quantity"] <= 0).sum())
print("Price <= 0:", (df["Price"] <= 0).sum())

Quantity <= 0: 0
Price <= 0: 70


In [ ]:
df = df[df["Price"] > 0]

print("Dataset size after removing invalid prices:", df.shape)

Dataset size after removing invalid prices: (779425, 8)


kiem tra stockcode

In [ ]:
print("Number of unique StockCodes:", df["StockCode"].nunique())
print("\nSample StockCodes:")
print(df["StockCode"].value_counts().head(20))

Number of unique StockCodes: 4631

Sample StockCodes:
StockCode
85123A    5023
22423     3335
85099B    3296
84879     2692
20725     2609
21212     2557
47566     2098
20727     2045
22383     2039
21034     1950
21232     1935
22382     1935
22384     1874
21754     1852
22139     1848
20914     1821
22469     1821
20728     1820
84991     1813
POST      1803
Name: count, dtype: int64


In [ ]:
special_stockcodes = df[
    ~df["StockCode"].astype(str).str.match(r"^\d+[A-Za-z]?$")
]["StockCode"].value_counts()

print("Special StockCodes:")
print(special_stockcodes.head(30))

Special StockCodes:
StockCode
POST            1803
15056BL          799
M                681
C2               248
79323LP          159
79323GR           76
ADJUST            32
BANK CHARGES      31
PADS              17
DOT               16
TEST001            9
D                  5
ADJUST2            3
SP1002             2
TEST002            1
Name: count, dtype: int64


In [ ]:
special_codes = [
    "POST", "15056BL", "M", "C2", "79323LP", "79323GR",
    "ADJUST", "BANK CHARGES", "PADS", "DOT",
    "TEST001", "D", "ADJUST2", "SP1002", "TEST002"
]

print(
    df[df["StockCode"].isin(special_codes)]
    [["StockCode", "Description"]]
    .drop_duplicates()
    .sort_values("StockCode")
)

           StockCode                          Description
613          15056BL              EDWARDIAN PARASOL BLACK
3173         79323GR                  GREEN CHERRY LIGHTS
572          79323LP             LIGHT PINK CHERRY LIGHTS
70976         ADJUST  Adjustment by john on 26/01/2010 16
71058         ADJUST  Adjustment by john on 26/01/2010 17
249672       ADJUST2  Adjustment by Peter on Jun 25 2010 
18466   BANK CHARGES                         Bank Charges
148098  BANK CHARGES                         Bank Charges
9292              C2                             CARRIAGE
160443             D                             Discount
842968           DOT                       DOTCOM POSTAGE
11310              M                               Manual
62299           PADS           PADS TO MATCH ALL CUSHIONS
89              POST                              POSTAGE
377066        SP1002               KID'S CHALKBOARD/EASEL
27994        TEST001              This is a test product.
39411        T

In [ ]:
non_product_codes = [
    "POST",
    "C2",
    "ADJUST",
    "ADJUST2",
    "BANK CHARGES",
    "DOT",
    "M",
    "D",
    "TEST001",
    "TEST002"
]

df = df[~df["StockCode"].isin(non_product_codes)]

print("Dataset size after removing non-product transactions:", df.shape)

Dataset size after removing non-product transactions: (776596, 8)


tạo biến Revenue = Quantity x Price

In [ ]:
df["Revenue"] = df["Quantity"] * df["Price"]

print("Revenue column created successfully.")
print(df[["Quantity", "Price", "Revenue"]].head())

Revenue column created successfully.
   Quantity  Price  Revenue
0        12   6.95     83.4
1        12   6.75     81.0
2        12   6.75     81.0
3        48   2.10    100.8
4        24   1.25     30.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv("//online_retail_cleaned.csv")

print("Dataset size:", df.shape)

Dataset size: (776596, 9)


In [ ]:
print(df["Customer ID"].dtype)
print(df["Customer ID"].head())

float64
0    13085.0
1    13085.0
2    13085.0
3    13085.0
4    13085.0
Name: Customer ID, dtype: float64


In [ ]:
df["Customer ID"] = df["Customer ID"].astype(int).astype(str)

print(df["Customer ID"].dtype)
print(df["Customer ID"].head())

object
0    13085
1    13085
2    13085
3    13085
4    13085
Name: Customer ID, dtype: object


In [ ]:
df.to_csv("online_retail_cleaned.csv", index=False)

print("Final cleaned dataset saved successfully.")

Final cleaned dataset saved successfully.


In [ ]:
from google.colab import files
files.download("online_retail_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>